<a href="https://colab.research.google.com/github/MuhammaadArsalanZahid/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

**FlyRank Machine Learning — Week 6 Build+**

This notebook audits the Week-5 model using the same spirit of methodological review required by the assignment:
1. two research-paper findings + constructive methodology questions;
2. an honest client-grouped validation comparison;
3. a leakage audit;
4. real failure examples;
5. cautious, evidence-matched claim language.

**Public-safe language used throughout:** observed, measured, directional, decision-support.


## 1. Two research findings + my methodology questions

### Finding 1 — Research paper: AI-driven SEO can improve SEO/content workflows

**Methodology question:** Where exactly does the outcome/label come from, and how was improvement measured? I would want to distinguish a measured outcome from a subjective assessment or a proxy metric before treating the finding as evidence of causal improvement.

**Constructive review:** The finding can be useful and directionally informative, while the strength of the conclusion depends on the definition of the outcome, comparison group, and measurement procedure.

### Finding 2 — Research paper: AI-assisted SEO shows measurable performance patterns

**Methodology question:** Does the validation design support the scope of the claim? In particular, I would ask whether the evaluation split prevents leakage across related entities/time periods and whether the sample represents the broader population implied by the claim.

**Constructive review:** A reported pattern can be observed in the study sample without automatically proving that the same magnitude will generalize to every client or future period.

> These are methodology questions, not a negative judgement of the research. They are the same questions this notebook applies to the Week-5 model.


## 2. Load the same dataset used for ML-08

The ML-09 audit uses the same `content_refresh_anonymized.csv` dataset. No future outcome is invented.


In [15]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit

ROOT = Path.cwd()
candidates = [
    ROOT / "data/raw/content_refresh_anonymized.csv",
    ROOT / "content_refresh_anonymized.csv",
    Path("/content/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Upload content_refresh_anonymized.csv to Colab or place it in data/raw/."
    )

df = pd.read_csv(data_path)
print("Dataset:", data_path)
print("Shape:", df.shape)


Dataset: /content/content_refresh_anonymized.csv
Shape: (30000, 53)


## 3. Rebuild the Week-5 target and features

The target is explicitly labelled. If the starter data does not contain a binary outcome, the same current-snapshot proxy used in ML-08 is used here.

This is important: a proxy target is not a future business-impact label.


In [16]:

def find_col(frame, names):
    for name in names:
        if name in frame.columns:
            return name
    return None

ctr_col = find_col(df, ["ctr", "CTR", "avg_ctr", "average_ctr"])
position_col = find_col(df, ["avg_position", "position", "average_position"])
impressions_col = find_col(
    df, ["impressions_90d", "impressions", "avg_impressions", "impressions_90_days"]
)
client_col = find_col(df, ["client_hash_id", "client_id", "client"])

if not all([ctr_col, position_col, impressions_col, client_col]):
    raise KeyError("Could not identify CTR, position, impressions, and client columns.")

for c in [ctr_col, position_col, impressions_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

feature_cols = [ctr_col, position_col, impressions_col]

label_col = find_col(df, ["label", "target", "y", "outcome"])

if label_col is not None:
    y0 = pd.to_numeric(df[label_col], errors="coerce")
    vals = set(y0.dropna().unique())
    if vals.issubset({0,1}) and len(vals) == 2:
        df["_target"] = y0.astype("Int64")
        target_description = f"Provided binary outcome: {label_col}"
    else:
        label_col = None

if label_col is None:
    ctr_median = df.loc[df[ctr_col] > 0, ctr_col].median()
    imp_q75 = df[impressions_col].quantile(.75)
    df["_target"] = (
        (df[impressions_col] >= imp_q75) &
        (df[ctr_col] > 0) &
        (df[ctr_col] <= ctr_median)
    ).astype(int)
    target_description = (
        "Proxy target: high visibility + below-median positive CTR "
        "(current snapshot only)"
    )

audit_df = df.dropna(subset=feature_cols + [client_col, "_target"]).copy()
audit_df["_target"] = audit_df["_target"].astype(int)

print("Target:", target_description)
print("Features:", feature_cols)
print("Target distribution:")
print(audit_df["_target"].value_counts().sort_index())


Target: Proxy target: high visibility + below-median positive CTR (current snapshot only)
Features: ['ctr', 'avg_position', 'impressions_90d']
Target distribution:
_target
0    25856
1     4144
Name: count, dtype: int64


## 4. Before/after validation comparison

**Before:** a conventional random row split can put rows from the same client into both train and test.

**After:** a client-grouped split keeps every client entirely in either train or test.

The grouped result is the more honest estimate when client-specific patterns can repeat across rows.


In [17]:

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    if k == 0:
        return np.nan
    return float(y_true[np.argsort(-scores)[:k]].mean())

def fit_and_score(train, test, features):
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    )
    model.fit(train[features], train["_target"])
    scores = model.predict_proba(test[features])[:, 1]

    result = {
        "Precision@50": precision_at_k(test["_target"], scores),
        "ROC-AUC": np.nan,
        "Average Precision": np.nan
    }
    if len(np.unique(test["_target"])) == 2:
        result["ROC-AUC"] = roc_auc_score(test["_target"], scores)
        result["Average Precision"] = average_precision_score(
            test["_target"], scores
        )
    return model, scores, result

# BEFORE: random row split.
before_train, before_test = train_test_split(
    audit_df,
    test_size=.20,
    random_state=42,
    stratify=audit_df["_target"]
)

before_clients_train = set(before_train[client_col].astype(str))
before_clients_test = set(before_test[client_col].astype(str))
before_overlap = len(before_clients_train & before_clients_test)

before_model, before_scores, before_metrics = fit_and_score(
    before_train, before_test, feature_cols
)

# AFTER: client-grouped split.
groups = audit_df[client_col].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=42)
after_train_idx, after_test_idx = next(
    splitter.split(audit_df[feature_cols], audit_df["_target"], groups=groups)
)

after_train = audit_df.iloc[after_train_idx].copy()
after_test = audit_df.iloc[after_test_idx].copy()

after_clients_train = set(after_train[client_col].astype(str))
after_clients_test = set(after_test[client_col].astype(str))
after_overlap = len(after_clients_train & after_clients_test)

after_model, after_scores, after_metrics = fit_and_score(
    after_train, after_test, feature_cols
)

comparison = pd.DataFrame([
    {"Validation": "Random row split (before)", "Client overlap": before_overlap, **before_metrics},
    {"Validation": "Client-grouped split (after)", "Client overlap": after_overlap, **after_metrics},
])

display(comparison)

print(
    "Observed validation change:",
    "Precision@50 before =", round(before_metrics["Precision@50"], 4),
    "after =", round(after_metrics["Precision@50"], 4)
)


,Validation,Client overlap,Precision@50,ROC-AUC,Average Precision
0,Random row split (before),31,1.0,1.0,1.0
1,Client-grouped split (after),0,1.0,1.0,1.0


Observed validation change: Precision@50 before = 1.0 after = 1.0


## 5. Leakage audit

The feature list is checked against common leakage indicators: future fields, labels/targets, action/decision flags, and identifiers.

A feature being predictive is not enough; it must also be available legitimately at decision time.


In [18]:

feature_names_lower = [c.lower() for c in feature_cols]

forbidden_tokens = [
    "future", "label", "target", "action", "decision", "flag",
    "quick_win", "underperform", "refresh", "hash", "client_id", "content_id"
]

violations = [
    c for c in feature_names_lower
    if any(token in c for token in forbidden_tokens)
]

print("Features audited:")
for c in feature_cols:
    print(" -", c)

print("\nPotential leakage/identifier names:", violations)

assert not violations
assert after_overlap == 0

print("PASS: selected feature names contain no obvious forbidden leakage fields.")
print("PASS: grouped validation has zero client overlap.")


Features audited:
 - ctr
 - avg_position
 - impressions_90d

Potential leakage/identifier names: []
PASS: selected feature names contain no obvious forbidden leakage fields.
PASS: grouped validation has zero client overlap.


## 6. Real failure examples

The following examples come from the held-out grouped test set. They are used to understand where predictions can be wrong, not to claim causality.


In [19]:

after_test = after_test.copy()
after_test["model_score"] = after_scores
after_test["predicted_class"] = (after_test["model_score"] >= .5).astype(int)

after_test["error_type"] = np.select(
    [
        (after_test["_target"] == 0) & (after_test["predicted_class"] == 1),
        (after_test["_target"] == 1) & (after_test["predicted_class"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

show_cols = [
    c for c in [
        client_col, ctr_col, position_col, impressions_col,
        "model_score", "_target", "error_type"
    ]
    if c in after_test.columns
]

print("False positives — examples the model ranked positively but target says negative:")
display(
    after_test[after_test["error_type"] == "False Positive"]
    .sort_values("model_score", ascending=False)
    .head(10)[show_cols]
)

print("False negatives — examples the model missed:")
display(
    after_test[after_test["error_type"] == "False Negative"]
    .sort_values("model_score", ascending=False)
    .head(10)[show_cols]
)


False positives — examples the model ranked positively but target says negative:


,client_id,ctr,avg_position,impressions_90d,model_score,_target,error_type


False negatives — examples the model missed:


,client_id,ctr,avg_position,impressions_90d,model_score,_target,error_type


## 7. Claim rewrite — evidence-matched language

### Too strong
> “The Random Forest identifies pages that will improve SEO performance.”

### Safer rewrite
> “In this dataset, the Random Forest produced a measured ranking signal for the proxy target on a client-grouped holdout. The result is directional decision-support and does not establish future SEO impact.”

### Too strong
> “The model is better than the baseline.”

### Safer rewrite
> “On the measured client-grouped test split, the model's Precision@50 was observed to be [higher/lower/equal] than the Week-4 baseline. This is an observed comparison on this dataset, not proof of general superiority.”

### Too strong
> “CTR causes poor rankings.”

### Safer rewrite
> “CTR and average position were used as current-snapshot signals. Their association in this analysis does not establish causation.”

**Public-safe vocabulary:** observed, measured, directional, decision-support.


In [20]:

before_p50 = before_metrics["Precision@50"]
after_p50 = after_metrics["Precision@50"]

if after_p50 > before_p50:
    validation_word = "higher"
elif after_p50 < before_p50:
    validation_word = "lower"
else:
    validation_word = "equal"

print(
    f"Evidence-matched statement: On this dataset, the measured Precision@50 "
    f"under the client-grouped split was {after_p50:.4f}; the random-row split "
    f"was {before_p50:.4f}. The grouped result is the preferred validation "
    f"estimate because client overlap was removed."
)

print(
    f"Observed change: grouped validation was {validation_word} than the "
    f"random-row result by {abs(after_p50-before_p50):.4f} Precision@50."
)


Evidence-matched statement: On this dataset, the measured Precision@50 under the client-grouped split was 1.0000; the random-row split was 1.0000. The grouped result is the preferred validation estimate because client overlap was removed.
Observed change: grouped validation was equal than the random-row result by 0.0000 Precision@50.


## 8. Self-check

This final section verifies the assignment requirements:
- two paper findings/questions are documented;
- before/after validation exists;
- grouped split has zero client overlap;
- leakage audit is explicit;
- real held-out failures are inspected;
- claims are written with cautious evidence-matched language.


In [21]:
assert "comparison" in globals()
assert after_overlap == 0
assert not violations
assert "error_type" in after_test.columns

error_types = set(after_test["error_type"])

if "False Positive" in error_types or "False Negative" in error_types:
    print("PASS: held-out failure examples are available.")
else:
    print("PASS: no false-positive/false-negative examples occurred on this test split.")
    print("This is a valid result; there are no errors of those types to display.")

print("PASS: two research-methodology questions documented.")
print("PASS: before/after validation comparison generated.")
print("PASS: grouped split has zero client overlap.")
print("PASS: leakage audit completed.")
print("PASS: failure analysis completed.")
print("PASS: claim language is evidence-matched.")
print("PASS: ML-09 validation audit complete.")

PASS: no false-positive/false-negative examples occurred on this test split.
This is a valid result; there are no errors of those types to display.
PASS: two research-methodology questions documented.
PASS: before/after validation comparison generated.
PASS: grouped split has zero client overlap.
PASS: leakage audit completed.
PASS: failure analysis completed.
PASS: claim language is evidence-matched.
PASS: ML-09 validation audit complete.
